<a href="https://colab.research.google.com/github/Moquiuti/LangGraph_Orquestrando_agentes_e_multiagentes/blob/main/Human_in_the_Loop_com_LangGraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q -U langgraph
!pip install -q -U langgraph-checkpoint-sqlite
!pip install -q -U langchain
!pip install -q -U langchain-core
!pip install -q -U langchain-google-genai
!pip install -q -U langchain-tavily
!pip install -q -U python-dotenv

In [2]:
import os
import uuid
import sqlite3
from typing import TypedDict, Annotated, Sequence

from dotenv import load_dotenv

from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    AIMessage,
    ToolMessage
)

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_tavily import TavilySearch

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver

In [3]:
try:
    from google.colab import userdata

    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

    print("API Keys carregadas pelos Secrets do Colab.")

except Exception:
    load_dotenv()
    print("Tentando carregar variáveis via arquivo .env.")

API Keys carregadas pelos Secrets do Colab.


In [4]:
if not os.getenv("GEMINI_API_KEY"):
    raise ValueError("GEMINI_API_KEY não encontrada.")

if not os.getenv("TAVILY_API_KEY"):
    raise ValueError("TAVILY_API_KEY não encontrada.")

print("Chaves carregadas com sucesso.")

Chaves carregadas com sucesso.


In [6]:
def garantir_id_mensagem(message: BaseMessage) -> BaseMessage:
    if not getattr(message, "id", None):
        message.id = str(uuid.uuid4())
    return message

In [7]:
def reduceMessages(left: Sequence[BaseMessage], right: Sequence[BaseMessage]):
    left = list(left or [])
    right = list(right or [])

    mensagens_por_id = {}

    for msg in left:
        msg = garantir_id_mensagem(msg)
        mensagens_por_id[msg.id] = msg

    ordem = [msg.id for msg in left]

    for msg in right:
        msg = garantir_id_mensagem(msg)

        if msg.id not in mensagens_por_id:
            ordem.append(msg.id)

        mensagens_por_id[msg.id] = msg

    return [mensagens_por_id[msg_id] for msg_id in ordem]

In [8]:
#teste rápido
msg1 = HumanMessage(content="Mensagem original", id="teste-1")
msg2 = HumanMessage(content="Mensagem alterada pelo humano", id="teste-1")
msg3 = AIMessage(content="Nova mensagem")

resultado_reduce = reduceMessages([msg1], [msg2, msg3])

for msg in resultado_reduce:
    print(msg.id, "->", msg.content)

teste-1 -> Mensagem alterada pelo humano
60ac33c6-3d98-48ff-9715-4c4a4dec67c0 -> Nova mensagem


In [9]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], reduceMessages]

In [10]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

search_tool = TavilySearch(
    max_results=1,
    topic="general"
)

tools = [search_tool]

In [11]:
class Agent:
    def __init__(self, model, tools, checkpointer):
        self.model = model.bind_tools(tools)
        self.tools = {tool.name: tool for tool in tools}

        graph = StateGraph(AgentState)

        graph.add_node("llm", self.call_llm)
        graph.add_node("action", self.take_action)

        graph.set_entry_point("llm")

        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {
                True: "action",
                False: END
            }
        )

        graph.add_edge("action", "llm")

        self.graph = graph.compile(
            checkpointer=checkpointer,
            interrupt_before=["action"]
        )

    def call_llm(self, state: AgentState):
        messages = list(state["messages"])

        response = self.model.invoke(messages)

        if not getattr(response, "id", None):
            response.id = str(uuid.uuid4())

        return {
            "messages": [response]
        }

    def exists_action(self, state: AgentState):
        result = state["messages"][-1]

        return hasattr(result, "tool_calls") and len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state["messages"][-1].tool_calls
        results = []

        for tool_call in tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            tool_id = tool_call["id"]

            print(f"\nExecutando ferramenta aprovada: {tool_name}")
            print(f"Argumentos aprovados: {tool_args}")

            if tool_name not in self.tools:
                result = f"Ferramenta {tool_name} não encontrada."
            else:
                result = self.tools[tool_name].invoke(tool_args)

            results.append(
                ToolMessage(
                    tool_call_id=tool_id,
                    name=tool_name,
                    content=str(result),
                    id=str(uuid.uuid4())
                )
            )

        return {
            "messages": results
        }

In [12]:
conn = sqlite3.connect(
    "hitl_langgraph.sqlite",
    check_same_thread=False
)

memory = SqliteSaver(conn)

In [13]:
agent = Agent(
    model=llm,
    tools=tools,
    checkpointer=memory
)

In [14]:
#ThreadId dinâmico
thread_id = f"hitl-{uuid.uuid4()}"

config = {
    "configurable": {
        "thread_id": thread_id
    }
}

print("ThreadId criado:", thread_id)




ThreadId criado: hitl-e505b284-92b1-4d3c-94c2-c6c7499c4bd0


In [15]:
pergunta = "Pesquise na internet o que é LangGraph e me traga um resumo curto."

eventos = agent.graph.stream(
    {
        "messages": [
            HumanMessage(
                content=pergunta,
                id=str(uuid.uuid4())
            )
        ]
    },
    config=config,
    stream_mode="values"
)

for evento in eventos:
    mensagens = evento.get("messages", [])
    if not mensagens:
        continue

    ultima = mensagens[-1]

    print("\n--- EVENTO ---")
    print("Tipo:", ultima.type)
    print("ID:", ultima.id)

    if hasattr(ultima, "tool_calls") and ultima.tool_calls:
        print("Tool calls solicitadas:")
        print(ultima.tool_calls)

    print("Conteúdo:")
    print(ultima.content)


--- EVENTO ---
Tipo: human
ID: 54091c41-5f4f-4db9-bd24-530ebbfe7d96
Conteúdo:
Pesquise na internet o que é LangGraph e me traga um resumo curto.

--- EVENTO ---
Tipo: ai
ID: lc_run--019e2c46-e6ce-7b51-ba96-717a8f9f1248-0
Tool calls solicitadas:
[{'name': 'tavily_search', 'args': {'query': 'o que é LangGraph'}, 'id': '66171d4e-19b6-481e-9327-1739faa12bc9', 'type': 'tool_call'}]
Conteúdo:



In [16]:
snapshot = agent.graph.get_state(config)

print("Próximo nó:", snapshot.next)
print("Quantidade de mensagens:", len(snapshot.values["messages"]))

Próximo nó: ('action',)
Quantidade de mensagens: 2


In [17]:
ultima_msg = snapshot.values["messages"][-1]

print("Tipo:", ultima_msg.type)
print("ID:", ultima_msg.id)
print("Conteúdo:", ultima_msg.content)
print("Tool calls:", getattr(ultima_msg, "tool_calls", None))

Tipo: ai
ID: lc_run--019e2c46-e6ce-7b51-ba96-717a8f9f1248-0
Conteúdo: 
Tool calls: [{'name': 'tavily_search', 'args': {'query': 'o que é LangGraph'}, 'id': '66171d4e-19b6-481e-9327-1739faa12bc9', 'type': 'tool_call'}]


In [18]:
for evento in agent.graph.stream(
    None,
    config=config,
    stream_mode="values"
):
    mensagens = evento.get("messages", [])
    if not mensagens:
        continue

    ultima = mensagens[-1]

    print("\n--- EVENTO APÓS APROVAÇÃO ---")
    print("Tipo:", ultima.type)
    print("Conteúdo:")
    print(ultima.content)


--- EVENTO APÓS APROVAÇÃO ---
Tipo: ai
Conteúdo:


Executando ferramenta aprovada: tavily_search
Argumentos aprovados: {'query': 'o que é LangGraph'}

--- EVENTO APÓS APROVAÇÃO ---
Tipo: tool
Conteúdo:
{'query': 'o que é LangGraph', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.datacamp.com/tutorial/langgraph-tutorial', 'title': 'LangGraph Tutorial: What Is LangGraph and How to Use It? | DataCamp', 'content': "# LangGraph Tutorial: What Is LangGraph and How to Use It? LangGraph is a library within the LangChain ecosystem that provides a framework for defining, coordinating, and executing multiple LLM agents (or chains) in a structured and efficient manner. Imagine you're building a complex, multi-agent large language model (LLM) application. It's exciting, but it comes with challenges: managing the state of various agents, coordinating their interactions, and handling errors effectively. LangGraph provides a framework for defining, coordin

In [19]:
thread_id_editado = f"hitl-editado-{uuid.uuid4()}"

config_editado = {
    "configurable": {
        "thread_id": thread_id_editado
    }
}

print("ThreadId criado:", thread_id_editado)

ThreadId criado: hitl-editado-6d499c01-8563-4426-9093-5ca7655b00cd


In [20]:
pergunta = "Pesquise na internet novidades sobre IA generativa no Brasil."

for evento in agent.graph.stream(
    {
        "messages": [
            HumanMessage(
                content=pergunta,
                id=str(uuid.uuid4())
            )
        ]
    },
    config=config_editado,
    stream_mode="values"
):
    mensagens = evento.get("messages", [])
    if not mensagens:
        continue

    ultima = mensagens[-1]

    print("\n--- EVENTO ---")
    print("Tipo:", ultima.type)
    print("ID:", ultima.id)

    if hasattr(ultima, "tool_calls") and ultima.tool_calls:
        print("Tool calls solicitadas:")
        print(ultima.tool_calls)

    print("Conteúdo:")
    print(ultima.content)


--- EVENTO ---
Tipo: human
ID: 37d6dcc8-a5f1-426e-923d-003b5cc16634
Conteúdo:
Pesquise na internet novidades sobre IA generativa no Brasil.

--- EVENTO ---
Tipo: ai
ID: lc_run--019e2c48-58cb-7720-a0ea-b4c64fb4e688-0
Tool calls solicitadas:
[{'name': 'tavily_search', 'args': {'query': 'IA generativa no Brasil', 'topic': 'news', 'time_range': 'month'}, 'id': 'd49ab43e-b799-4207-a09d-595901147bd5', 'type': 'tool_call'}]
Conteúdo:



In [21]:
snapshot_editado = agent.graph.get_state(config_editado)

ultima_msg = snapshot_editado.values["messages"][-1]

print("ID da última mensagem:", ultima_msg.id)
print("Tool calls originais:")
print(ultima_msg.tool_calls)

ID da última mensagem: lc_run--019e2c48-58cb-7720-a0ea-b4c64fb4e688-0
Tool calls originais:
[{'name': 'tavily_search', 'args': {'query': 'IA generativa no Brasil', 'topic': 'news', 'time_range': 'month'}, 'id': 'd49ab43e-b799-4207-a09d-595901147bd5', 'type': 'tool_call'}]


In [22]:
#Injetei manualmente uma resposta modificada no estado
mensagem_editada = ultima_msg.copy(deep=True)

mensagem_editada.tool_calls[0]["args"]["query"] = (
    "LangGraph human in the loop interrupt_before action Python"
)

mensagem_editada.content = (
    "Ação revisada por humano antes da execução da ferramenta."
)

print("Tool calls após edição humana:")
print(mensagem_editada.tool_calls)

Tool calls após edição humana:
[{'name': 'tavily_search', 'args': {'query': 'LangGraph human in the loop interrupt_before action Python', 'topic': 'news', 'time_range': 'month'}, 'id': 'd49ab43e-b799-4207-a09d-595901147bd5', 'type': 'tool_call'}]


/tmp/ipykernel_1410/3786714001.py:2: PydanticDeprecatedSince20: The `copy` method is deprecated; use `model_copy` instead. See the docstring of `BaseModel.copy` for details about how to handle `include` and `exclude`. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  mensagem_editada = ultima_msg.copy(deep=True)


In [23]:
agent.graph.update_state(
    config_editado,
    {
        "messages": [mensagem_editada]
    }
)

{'configurable': {'thread_id': 'hitl-editado-6d499c01-8563-4426-9093-5ca7655b00cd',
  'checkpoint_ns': '',
  'checkpoint_id': '1f150740-76b2-6f95-8002-ae5e4d152d45'}}

In [24]:
#Confirmando se o estado foi alterado
snapshot_pos_edicao = agent.graph.get_state(config_editado)
ultima_pos_edicao = snapshot_pos_edicao.values["messages"][-1]

print("Conteúdo após edição:")
print(ultima_pos_edicao.content)

print("Tool calls após edição:")
print(ultima_pos_edicao.tool_calls)

Conteúdo após edição:
Ação revisada por humano antes da execução da ferramenta.
Tool calls após edição:
[{'name': 'tavily_search', 'args': {'query': 'LangGraph human in the loop interrupt_before action Python', 'topic': 'news', 'time_range': 'month'}, 'id': 'd49ab43e-b799-4207-a09d-595901147bd5', 'type': 'tool_call'}]


In [25]:
for evento in agent.graph.stream(
    None,
    config=config_editado,
    stream_mode="values"
):
    mensagens = evento.get("messages", [])
    if not mensagens:
        continue

    ultima = mensagens[-1]

    print("\n--- EVENTO APÓS INTERVENÇÃO HUMANA ---")
    print("Tipo:", ultima.type)

    if hasattr(ultima, "tool_calls") and ultima.tool_calls:
        print("Tool calls:", ultima.tool_calls)

    print("Conteúdo:")
    print(ultima.content)


--- EVENTO APÓS INTERVENÇÃO HUMANA ---
Tipo: ai
Tool calls: [{'name': 'tavily_search', 'args': {'query': 'LangGraph human in the loop interrupt_before action Python', 'topic': 'news', 'time_range': 'month'}, 'id': 'd49ab43e-b799-4207-a09d-595901147bd5', 'type': 'tool_call'}]
Conteúdo:
Ação revisada por humano antes da execução da ferramenta.

Executando ferramenta aprovada: tavily_search
Argumentos aprovados: {'query': 'LangGraph human in the loop interrupt_before action Python', 'topic': 'news', 'time_range': 'month'}

--- EVENTO APÓS INTERVENÇÃO HUMANA ---
Tipo: tool
Conteúdo:
{'query': 'LangGraph human in the loop interrupt_before action Python', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://medium.com/algomart/human-in-the-loop-in-langgraph-how-to-pause-review-and-resume-an-ai-workflow-b3b96f447c83', 'title': 'Human-in-the-Loop in LangGraph: How to Pause, Review ... - Medium', 'content': '1. Letting the Agent Execute Before Review. The who

In [26]:
#testando a continuidade da thread
for evento in agent.graph.stream(
    {
        "messages": [
            HumanMessage(
                content="Com base na pesquisa anterior, explique em uma frase qual foi o ponto principal.",
                id=str(uuid.uuid4())
            )
        ]
    },
    config=config_editado,
    stream_mode="values"
):
    mensagens = evento.get("messages", [])
    if not mensagens:
        continue

    ultima = mensagens[-1]

    print("\n--- EVENTO DE CONTINUIDADE ---")
    print("Tipo:", ultima.type)
    print("Conteúdo:")
    print(ultima.content)


--- EVENTO DE CONTINUIDADE ---
Tipo: human
Conteúdo:
Com base na pesquisa anterior, explique em uma frase qual foi o ponto principal.

--- EVENTO DE CONTINUIDADE ---
Tipo: ai
Conteúdo:
[{'type': 'text', 'text': 'Não foi possível determinar o ponto principal sobre IA generativa no Brasil, pois os resultados da pesquisa para essa consulta específica não foram fornecidos.', 'extras': {'signature': 'CtkGAQw51semr8DHQAlzrVtIk5o4qzaxDP71PW71y0BGcM21JnkeADo0e2JrPwYnCIRJTMz/fHaZ/WNs+fOp2iN+czCNHBrKGDyUEnvp05snQfiVUtrM1K+KttEizYdF/gzJf93+MJPBju2HMVlqckk9QcLXrhyLTOD+o2zRYOBjQsoHcIukRE1IqB1axAEHT/Q4G+aftQlm1TckXNEvV1HGLgZvWUCiG0m7M90ehVoI7qETic8Wmvnzc9stwwR/Ei2owWt6mMl6zLHkSx3umJZ4pXW/i1Jk7MK6K3YgrBWFPTJFzf3Z797hOUVpJKiOYJmAIqpZAnwsGqZdm8GDDKTfuh0wJn58JMqyJXqRcCjlUyxGkfnc4Dh0Ri4I9+YrCwgMI1JxBu7o+3Ir8kq+wJUJ8z7K/wtf57zpklib6UvMEN+IPwS+H2pbN8Age48WwM4G5D4H1z8xX0cxQIEOOmU+FkKNsvXae4/9LcaMTJ9Ebd+f3KwV4H9kGe/p3jXyAWOZ22P9urJd+++JKpfjJq12uL8oVnf+gGRAMThsMZ1cIpQG5L4ad3kl1y12amJy10oI5za0TE6HJzZTaLzXCAMS

In [27]:
from google.colab import files

files.download("hitl_langgraph.sqlite")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>